# Milestone 4 Lab, notebook 7 of 7: Two experiments

## Where we are

| Notebook | What you do |
|---|---|
| 1. A figure, traced by hand | Find every record behind one figure yourself, and add them up |
| 2. Run records | Give every pipeline run an id and a record, and keep every run |
| 3. Lineage | Save which records made each figure, and prove it adds up |
| 4. Two runs compared | Find what changed between two runs: settings, inputs, figures, records |
| 5. A trace of the analyst | Give the analyst the new tools, and record every step it takes |
| 6. Scoring an agent | Measure an agent with a confusion matrix, kappa, and the cost of mistakes |
| **7. Two experiments** (you are here) | Run two versions of the analyst on the same questions, and compare the scores |

Model requests in this notebook: **about 40**. A full run of this lab costs about 5 US cents.

## What this notebook is for

In notebook 5 the analyst answered one new question well. In notebook 6 you learned how to score an agent properly. Now put them together.

You run **two versions** of the analyst on the same questions and compare them:

- **Without the run tool:** six tools. It has no record of earlier runs.
- **With the run tool:** seven tools. `compare_runs` is added.

A run like this, with scores, is called an **experiment**. Langfuse stores each experiment, with a trace for every answer, so you can compare them side by side.

Two questions are being asked. Does the new tool help with the new questions? And did adding it make the **old** questions worse? In milestone 3, small additions did.

> This notebook needs `OPENROUTER_API_KEY` and the three Langfuse settings. By default it runs five questions per experiment. Each experiment takes about a minute.

In [ ]:
import pandas as pd
from lab_helpers import connect_langfuse, show_doc, show_matrix

## Step 1: Start from the real history, and connect

In [ ]:
from retail_profit_stack.make_history import make_history

for run in make_history():
    print(run.run_id, "-", run.record["note"])

In [ ]:
langfuse = connect_langfuse()

## Step 2: The design

In [ ]:
show_doc("HLD.md", "10. The two experiments")

Langfuse has three words for this. A **dataset** is a stored list of questions with their correct answers. An **experiment** runs one version of the analyst on every question of a dataset. A **score** is a number attached to one answer or to a whole experiment.

## Step 3: The dataset

These are the fourteen evaluation questions: ten from earlier milestones and four new ones.

In [ ]:
from retail_profit_stack.evaluation import load_cases
from retail_profit_stack.scoring import right_level

all_cases = load_cases()

rows = []
for case in all_cases:
    rows.append({
        "case_id": case.case_id,
        "tiny": case.tiny,
        "right_level": right_level(case),
        "question": case.question[:90],
    })

pd.set_option("display.max_colwidth", 100)
pd.DataFrame(rows)

`upload_questions` stores them in Langfuse as a dataset called `profit-questions`. Each item's id comes from the question's id, so running this cell again updates the items and adds none.

In [ ]:
from retail_profit_stack.experiments import DATASET, upload_questions

upload_questions(all_cases)

dataset = langfuse.get_dataset(DATASET)
print(len(dataset.items), "questions in the dataset", DATASET)

Here is one item as Langfuse holds it: the input, and the expected output.

In [ ]:
items_by_id = {item.metadata["case_id"]: item for item in dataset.items}
item = items_by_id["E12"]

print("Input:   ", item.input)
print("Expected:", item.expected_output)

In Langfuse, open **Datasets** and then `profit-questions` to see them.

## Step 4: How many questions, how many times

`TINY = True` runs the five questions marked tiny. Set it to `False` to run all fourteen, which takes about four minutes per experiment and costs about three US cents.

`N_TIMES` is how many times each experiment runs. An agent does not give the same answer twice, so more runs give steadier scores.

In [ ]:
TINY = True
N_TIMES = 1

cases = load_cases(tiny=TINY)

print("Running", len(cases), "questions:", [case.case_id for case in cases])

## Step 5: One question, by hand

An experiment does the same thing for every question. Do it once yourself first.

`answer_and_score` asks the analyst one question, scores the answer against the question's correct answer, and returns one row of scores. It is three lines: `run_analyst`, then `score_case` from milestone 1, then `score_row` from notebook 6.

In [ ]:
from retail_profit_stack.experiments import answer_and_score

cases_by_id = {case.case_id: case for case in cases}
change_case = cases_by_id["E12"]
print(change_case.question)

row = answer_and_score(change_case, run_tool=True)
row

Read the row. `right_level` is the level a correct answer has, and `level` is what the analyst gave. `right` says whether every check passed. `mistake` and `mistake_cost` come from the cost table. `verified` says whether the verifier passed the answer.

Now the same question for the analyst **without** the run tool.

In [ ]:
row = answer_and_score(change_case, run_tool=False)
row

Without the tool the analyst has no record of earlier runs. The honest thing is to decline, which costs $200: a person has to answer. If it answered anyway, look at the headline: it had nothing to go on.

## Step 6: Experiment 1, without the run tool

`run_experiment` hands Langfuse three things: the questions, a **task** (the function to run for each question, which is `answer_and_score`), and **evaluators** (functions that turn each row into scores). Langfuse runs the task on each question, one at a time, and keeps a trace of each.

In [ ]:
from retail_profit_stack.experiments import run_experiment

without_tool = run_experiment(
    name="Without the run tool",
    description="The analyst has six tools and no record of earlier runs",
    cases=cases,
    run_tool=False,
    times=N_TIMES,
)

without_tool[["case_id", "right_level", "level", "right", "mistake", "mistake_cost", "model_cost"]]

In [ ]:
from retail_profit_stack.scoring import confusion_matrix, run_scores

show_matrix(confusion_matrix(without_tool), "Without the run tool")

## Step 7: Experiment 2, with the run tool

One argument differs.

In [ ]:
with_tool = run_experiment(
    name="With the run tool",
    description="The analyst has seven tools, including compare_runs",
    cases=cases,
    run_tool=True,
    times=N_TIMES,
)

with_tool[["case_id", "right_level", "level", "right", "mistake", "mistake_cost", "model_cost"]]

In [ ]:
show_matrix(confusion_matrix(with_tool), "With the run tool")

## Step 8: Compare the two

`run_scores` gives the scores of a whole experiment.

In [ ]:
scores = pd.DataFrame(
    [run_scores(without_tool), run_scores(with_tool)],
    index=["Without the run tool", "With the run tool"],
)
scores

How to read it:

- **accuracy** is the share of questions where every check passed. **level_accuracy** and **kappa** look only at the level.
- **cost_of_mistakes** is what the wrong answers would cost the business.
- **verified_rate** is the share of answers the verifier passed.
- **model_cost** and **seconds** are what the experiment itself cost.

Then check the second question from the top of this notebook. Look at the old questions (E1 to E10) in the two tables above. Did the same ones pass in both experiments?

Scores do not tell you **why**. For that, read the answers that were wrong.

In [ ]:
wrong = with_tool[~with_tool["right"]]

wrong[["case_id", "mistake", "failed_checks", "headline"]]

In [ ]:
wrong = without_tool[~without_tool["right"]]

wrong[["case_id", "mistake", "failed_checks", "headline"]]

If a table is empty, that experiment got everything right this time. Run it again and it may not: the analyst varies.

**A warning about small numbers.** With five questions, one answer moves accuracy by twenty points. Do not choose between two versions on one run of five questions. Set `TINY = False` and `N_TIMES = 3` before you trust a difference.

## Step 9: See it in Langfuse

Open Langfuse and look at three places:

- **Datasets → profit-questions → Runs.** Each experiment you ran is listed with its scores. Select two and compare them question by question.
- **Click one question in a run.** It opens the trace: every step the analyst took for that answer. This is how you find out why an answer was wrong.
- **Tracing.** Every trace, newest first, each with its `right` and `mistake_cost` scores.

When the lab was built, the full fourteen questions were run several times. What those runs showed is in the LLD, section 13.

## Step 10: Where this milestone leaves us

| | Before this milestone | After |
|---|---|---|
| Where did this number come from? | Five sample records | Every record, from lineage, proved to add up |
| Why did it change? | A guess | Looked up: the setting, input, or code that changed, and the records |
| Old figures | Overwritten | Kept, one folder per run |
| What did the analyst do? | A tool log, gone when the program ends | A trace in Langfuse, kept and shareable |
| Is the analyst better now? | One-off scripts | An experiment, with scores that can be compared |

## Step 11: From the lab to production

Each notebook's idea is rebuilt in the production package, with tests.

In [ ]:
show_doc("LLD.md", "13. Where each part is built")

To run the tests, open a terminal in this milestone's folder and run:

```
uv run pytest
```

The tests need no key, no network, and no Langfuse. They use a scripted fake model.

## Your assignment

1. **Run the full set.** Set `TINY = False` and `N_TIMES = 2`, and run both experiments again. Which questions fail in both versions? Open one failed answer's trace in Langfuse and find the step where it went wrong.
2. **Change the cost of a mistake.** In `retail_profit_stack/scoring.py`, change `too_confident` from 5000 to 500. Score the same tables again with `run_scores`. Does the ranking of the two versions change? What does that tell you about agreeing costs first?
3. **A third experiment.** Set `MODEL_NAME` in your `.env` file to another OpenRouter model, restart the notebook, and run "With the run tool" again under a new name. Compare accuracy, cost, and time with the default model in Langfuse.

## Cleaning up Langfuse

This last step is optional. Set `CLEAN_UP = True` to delete **every trace in the Langfuse project** your keys belong to, and the questions in the `profit-questions` dataset. This cannot be undone. Only do it in a project you made for this lab.

In [ ]:
CLEAN_UP = False

if CLEAN_UP:
    trace_ids = set()
    cursor = None
    while True:
        page = langfuse.api.observations.get_many(fields="core", limit=1000, cursor=cursor)
        trace_ids.update(observation.trace_id for observation in page.data)
        cursor = page.meta.cursor
        if not cursor:
            break

    trace_ids = list(trace_ids)
    for start in range(0, len(trace_ids), 1000):
        langfuse.api.trace.delete_multiple(trace_ids=trace_ids[start : start + 1000])

    items = langfuse.get_dataset(DATASET).items
    for item in items:
        langfuse.api.dataset_items.delete(id=item.id)

    print("Deleted", len(trace_ids), "traces and", len(items), "questions")

## What we now know

- An **experiment** runs one version of an agent on a fixed list of questions and scores every answer.
- Without a record of earlier runs, the analyst cannot say why a figure changed. With `compare_runs`, it can.
- An experiment also checks the questions that already worked. A new tool is only an improvement if the old answers stay right.
- Scores say **which** version is better. Traces say **why** an answer was wrong.
- Scores from a few questions are rough. Run more questions, more than once, before deciding.

You have finished milestone 4. Open `milestone_4_recap.html` in a browser to see what you built.